[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        B, S_q, D = x_q.shape
        B, S_kv, D = x_kv.shape
        assert D == self.d_model

        print('x_q:', x_q.shape)
        print('x_kv:', x_kv.shape)
        Q = self.W_q(x_q).reshape(B, S_q, self.num_heads, self.d_head).transpose(1, 2)
        print('Q:', Q.shape)
        K = self.W_k(x_kv).reshape(B, S_kv, self.num_heads, self.d_head).transpose(1, 2)
        print('K:', K.shape)
        V = self.W_v(x_kv).reshape(B, S_kv, self.num_heads, self.d_head).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)
        print(attn_scores.shape)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        print(attn_probs.shape)
        attn_output = torch.matmul(attn_probs, V)
        print(attn_output.shape)
        attn_output = attn_output.transpose(1, 2).reshape(B, S_q, self.d_model)
        print(attn_output.shape)
        return self.W_o(attn_output)


In [9]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

x_q: torch.Size([2, 6, 64])
x_kv: torch.Size([2, 10, 64])
Q: torch.Size([2, 4, 6, 16])
K: torch.Size([2, 4, 10, 16])
torch.Size([2, 4, 6, 10])
torch.Size([2, 4, 6, 10])
torch.Size([2, 4, 6, 16])
torch.Size([2, 6, 64])
Output: torch.Size([2, 6, 64])


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.5ms)
  ✅ [2/4] Q and KV different lengths (1.3ms)
  ✅ [3/4] No causal mask — all KV affects all Q (42.0ms)
  ✅ [4/4] Gradient flow (23.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (69.7ms total)
  Progress saved. Run status() to see your dashboard.

